# OMI Transactions Exploration

Exploratory analysis of OMI normalized transaction volumes (NTN) for Italian municipalities, 2011–2025.

The notebook is release-agnostic: it discovers annual files from their filenames, harmonizes year-specific schemas, validates keys before joins, and builds one municipality-year analytical panel for downstream analysis.

## 1. Setup and release discovery

The project root is detected automatically. File parsing uses the explicit OMI pattern `YYYY_LISTA-COM.csv`, `YYYY_VALORI-RES.csv`, `YYYY_VALORI-COM.csv`, `YYYY_VALORI-PER.csv`, avoiding fragile filename slicing.

In [ ]:
from pathlib import Path
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

EXPECTED_TABLES = {'LISTA-COM', 'VALORI-RES', 'VALORI-COM', 'VALORI-PER'}
FILENAME_RE = re.compile(r'^(?P<year>\d{4})_(?P<table>LISTA-COM|VALORI-RES|VALORI-COM|VALORI-PER)\.csv$', re.I)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'transactions').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/transactions. Run the notebook from the repository.')

PROJECT_ROOT = find_project_root()
TRANSACTIONS_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
print(f'Project root: {PROJECT_ROOT}')
print(f'Transactions root: {TRANSACTIONS_ROOT}')

In [ ]:
def discover_releases(root):
    records = []
    for path in root.rglob('*.csv'):
        match = FILENAME_RE.match(path.name)
        if match:
            records.append({'year': int(match.group('year')), 'table': match.group('table').upper(), 'path': path})
    catalogue = pd.DataFrame(records)
    if catalogue.empty:
        raise FileNotFoundError(f'No OMI transaction releases found under {root}')
    duplicates = catalogue.duplicated(['year', 'table'], keep=False)
    if duplicates.any():
        raise ValueError('Duplicate year/table combinations found:\n' + catalogue.loc[duplicates].to_string(index=False))
    coverage = catalogue.groupby('year')['table'].agg(lambda x: sorted(set(x))).rename('tables').reset_index()
    coverage['missing_tables'] = coverage['tables'].map(lambda x: sorted(EXPECTED_TABLES - set(x)))
    incomplete = coverage[coverage['missing_tables'].map(bool)]
    if not incomplete.empty:
        print('Warning: incomplete releases detected:')
        print(incomplete.to_string(index=False))
    return catalogue.sort_values(['year', 'table']).reset_index(drop=True)

catalogue = discover_releases(TRANSACTIONS_ROOT)
years = sorted(catalogue['year'].unique())
print(f'Releases discovered: {min(years)}–{max(years)} ({len(years)} years)')
display(catalogue[['year', 'table']].head(20))

## 2. Efficient schema harmonization and ingestion

Historical releases use small naming variations (`AREA`/`Area`, `prov`/`Provincia`, year-prefixed NTN fields). Columns are canonicalized and numeric NTN fields are parsed with the Italian decimal convention. The loader caches each release and concatenates once per table.

In [ ]:
def normalize_label(value):
    value = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode()
    value = re.sub(r'^\d{4}_', '', value.strip(), flags=re.I)
    value = re.sub(r'^ntn_?\d{4}_?', 'ntn_', value, flags=re.I)
    value = re.sub(r'\s+', '_', value)
    value = re.sub(r'[^0-9A-Za-z_]+', '_', value)
    return re.sub(r'_+', '_', value).strip('_').lower()

def normalize_columns(df):
    df = df.copy()
    df.columns = [normalize_label(c) for c in df.columns]
    return df

def find_column(columns, *patterns):
    for pattern in patterns:
        regex = re.compile(pattern, re.I)
        matches = [c for c in columns if regex.search(c)]
        if matches:
            return matches[0]
    return None

def parse_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    return pd.to_numeric(series.astype('string').str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

def read_omi_csv(path):
    return pd.read_csv(path, sep=';', dtype='string', na_values=['', 'NA', 'N/A', 'nan', '-'], keep_default_na=True)

def load_release_table(path, year, table):
    df = normalize_columns(read_omi_csv(path))
    codcom = find_column(df.columns, r'^codcom$', r'cod_com')
    if codcom is None:
        raise ValueError(f'{path.name}: municipality code column not found')
    df = df.rename(columns={codcom: 'codcom'})
    df['codcom'] = df['codcom'].astype('string').str.strip().str.upper()
    df['year'] = year
    df['table'] = table
    for column in df.columns:
        if column.startswith('ntn'):
            df[column] = parse_numeric(df[column])
    return df

def table_path(year, table):
    match = catalogue[(catalogue.year == year) & (catalogue.table == table)]
    if match.empty:
        raise FileNotFoundError(f'Missing {year}_{table}.csv')
    return Path(match.iloc[0].path)

_TABLE_CACHE = {}
def get_table(year, table):
    key = (year, table)
    if key not in _TABLE_CACHE:
        _TABLE_CACHE[key] = load_release_table(table_path(year, table), year, table)
    return _TABLE_CACHE[key]

tables = {}
for table in sorted(EXPECTED_TABLES):
    available = [y for y in years if not catalogue[(catalogue.year == y) & (catalogue.table == table)].empty]
    if available:
        tables[table] = pd.concat([get_table(y, table) for y in available], ignore_index=True, copy=False)

for table, df in tables.items():
    print(f'{table:10s}: {len(df):>8,} rows × {df.shape[1]:>2} columns')

## 3. Municipality dimension and residential panel

`LISTA-COM` is the descriptive dimension. The historical `prov` and newer `Provincia` fields are coalesced, while the municipality-year key is explicitly validated.

In [ ]:
lista = tables['LISTA-COM'].copy()
dimension_candidates = ['codcom', 'comune', 'area', 'regione', 'provincia', 'prov', 'cap', 'tag_mercato']
dimension_columns = [c for c in dimension_candidates if c in lista.columns]
municipality_dim = lista[dimension_columns].sort_values(['codcom', 'year']).drop_duplicates(['year', 'codcom'], keep='last').reset_index(drop=True)

if 'provincia' in municipality_dim.columns and 'prov' in municipality_dim.columns:
    municipality_dim['provincia'] = municipality_dim['provincia'].fillna(municipality_dim['prov'])
    municipality_dim = municipality_dim.drop(columns='prov')
elif 'provincia' not in municipality_dim.columns and 'prov' in municipality_dim.columns:
    municipality_dim = municipality_dim.rename(columns={'prov': 'provincia'})

for c in ['area', 'regione', 'provincia', 'comune', 'cap', 'tag_mercato']:
    if c in municipality_dim.columns:
        municipality_dim[c] = municipality_dim[c].astype('string').str.strip()
assert municipality_dim.duplicated(['year', 'codcom']).sum() == 0

res = tables['VALORI-RES'].copy()
size_patterns = {
    'ntn_upto_50': r'^ntn.*fino_a_50',
    'ntn_50_85': r'^ntn.*50.*85',
    'ntn_85_115': r'^ntn.*85.*115',
    'ntn_115_145': r'^ntn.*115.*145',
    'ntn_over_145': r'^ntn.*oltre_145',
}
size_columns = {name: find_column(res.columns, pattern) for name, pattern in size_patterns.items()}
size_columns = {k: v for k, v in size_columns.items() if v is not None}
total_column = find_column(res.columns, r'^ntn_?$')
if total_column is None:
    raise ValueError('Residential total NTN column not found')
res_panel = res[['year', 'codcom', *size_columns.values(), total_column]].rename(columns={**{v: k for k, v in size_columns.items()}, total_column: 'ntn_res'})
size_fields = list(size_columns)
res_panel['ntn_res_size_sum'] = res_panel[size_fields].sum(axis=1, min_count=1)
res_panel['ntn_res_reconciliation_gap'] = res_panel['ntn_res'] - res_panel['ntn_res_size_sum']

panel = res_panel.merge(municipality_dim, on=['year', 'codcom'], how='left', validate='many_to_one', indicator=True)
print(f"Unmatched municipality-year rows: {panel._merge.eq('left_only').sum():,}")
panel = panel.drop(columns='_merge')
assert panel.duplicated(['year', 'codcom']).sum() == 0
for c in ['area', 'regione', 'provincia', 'comune', 'cap', 'tag_mercato']:
    if c in panel.columns:
        panel[c] = panel[c].astype('category')
display(panel.head())

## 4. National and regional residential dynamics

All aggregates now derive from the same municipality-year panel, eliminating repeated raw-data joins.

In [ ]:
national = panel.groupby('year', as_index=False, observed=True).agg(
    ntn_res=('ntn_res', 'sum'),
    ntn_res_size_sum=('ntn_res_size_sum', 'sum'),
    municipalities=('codcom', 'nunique'),
).sort_values('year')
national['size_gap'] = national['ntn_res'] - national['ntn_res_size_sum']
national['yoy_pct'] = national['ntn_res'].pct_change().mul(100)
display(national.style.format({'ntn_res': '{:,.0f}', 'ntn_res_size_sum': '{:,.0f}', 'size_gap': '{:,.2f}', 'yoy_pct': '{:,.1f}%'}))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(national.year, national.ntn_res, marker='o')
ax.set(title='Italy — Residential NTN', xlabel='Year', ylabel='NTN')
ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

regional = panel.dropna(subset=['regione']).groupby(['year', 'regione'], as_index=False, observed=True).agg(ntn_res=('ntn_res', 'sum'))
regional['yoy_pct'] = regional.sort_values(['regione', 'year']).groupby('regione', observed=True).ntn_res.pct_change().mul(100)
latest_year = int(regional.year.max())
top_regions = regional[regional.year == latest_year].nlargest(10, 'ntn_res')
display(top_regions.style.format({'ntn_res': '{:,.0f}'}))

fig, ax = plt.subplots(figsize=(12, 6))
for region in top_regions.regione:
    s = regional[regional.regione == region]
    ax.plot(s.year, s.ntn_res, marker='o', label=region)
ax.set(title='Residential NTN — Main regions', xlabel='Year', ylabel='NTN')
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 5. Residential size mix

The five official residential size bands are reshaped into a compact time series. The official total NTN remains the authoritative volume measure; the band sum is a reconciliation check.

In [ ]:
size_long = panel.groupby('year', as_index=False, observed=True)[size_fields].sum().melt(id_vars='year', var_name='size_band', value_name='ntn')
size_long['share_pct'] = size_long.ntn / size_long.groupby('year', observed=True).ntn.transform('sum') * 100
display(size_long[size_long.year == latest_year].sort_values('share_pct', ascending=False).style.format({'ntn': '{:,.0f}', 'share_pct': '{:,.1f}%'}))

mix = size_long.pivot(index='year', columns='size_band', values='share_pct')
ax = mix.plot(kind='area', stacked=True, figsize=(12, 6))
ax.set(title='Residential NTN — Size mix', xlabel='Year', ylabel='Share (%)')
ax.legend(title='Size band', ncol=2); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 6. Non-residential reconciliation

`VALORI-COM` and `VALORI-PER` are aggregated through their normalized `ntn_*` fields, without hard-coding every historical category name.

In [ ]:
non_res_frames = []
for table in ['VALORI-COM', 'VALORI-PER']:
    if table not in tables:
        continue
    df = tables[table]
    ntn_columns = [c for c in df.columns if c.startswith('ntn_')]
    if not ntn_columns:
        continue
    frame = df[['year', 'codcom', *ntn_columns]].copy()
    frame['ntn_non_res'] = frame[ntn_columns].sum(axis=1, min_count=1)
    non_res_frames.append(frame[['year', 'codcom', 'ntn_non_res']])

non_res = pd.concat(non_res_frames, ignore_index=True) if non_res_frames else pd.DataFrame(columns=['year', 'codcom', 'ntn_non_res'])
non_res = non_res.groupby(['year', 'codcom'], as_index=False).agg(ntn_non_res=('ntn_non_res', 'sum'))

market_panel = panel.merge(non_res, on=['year', 'codcom'], how='left', validate='one_to_one')
market_panel['ntn_non_res'] = market_panel['ntn_non_res'].fillna(0)

reconciliation = market_panel.groupby('year', as_index=False, observed=True).agg(ntn_res=('ntn_res', 'sum'), ntn_non_res=('ntn_non_res', 'sum'))
reconciliation['ntn_all_loaded'] = reconciliation.ntn_res + reconciliation.ntn_non_res
display(reconciliation.style.format({'ntn_res': '{:,.0f}', 'ntn_non_res': '{:,.0f}', 'ntn_all_loaded': '{:,.0f}'}))

## 7. Final municipality-year analytical panel

This compact object is the hand-off for downstream price-volume and market-dynamics notebooks. It is built once and avoids reloading and rejoining the raw releases.

In [ ]:
panel_columns = ['year', 'codcom', *[c for c in ['comune', 'provincia', 'regione', 'area', 'cap', 'tag_mercato'] if c in market_panel.columns], 'ntn_res', *size_fields, 'ntn_non_res']
market_panel = market_panel[panel_columns].sort_values(['year', 'regione', 'provincia', 'comune', 'codcom']).reset_index(drop=True)

assert not market_panel.duplicated(['year', 'codcom']).any()
assert market_panel['ntn_res'].ge(0).all()
assert market_panel['codcom'].notna().all()

print(f'Final municipality-year panel: {len(market_panel):,} rows')
print(f'Unique municipalities: {market_panel.codcom.nunique():,}')
print(f'Years: {market_panel.year.min()}–{market_panel.year.max()}')
display(market_panel.head(10))

## 8. Data-quality checks

The notebook fails early if the release catalogue, municipality-year uniqueness, municipality keys or non-negative residential NTN assumptions are violated.

In [ ]:
checks = pd.Series({
    'release_catalogue_not_empty': not catalogue.empty,
    'years_detected': len(years) >= 1,
    'municipality_year_unique': not market_panel.duplicated(['year', 'codcom']).any(),
    'residential_non_negative': bool(market_panel.ntn_res.ge(0).all()),
    'no_missing_municipality_code': bool(market_panel.codcom.notna().all()),
}, name='passed')
display(checks.to_frame())
if not checks.all():
    raise AssertionError(f'Data-quality checks failed: {checks[~checks].index.tolist()}')
print('All data-quality checks passed.')